# Amazon Top-K · UltraGCN & SimpleX Long-Horizon Ablation + 6-Model Ensemble

## 이번 수정 사항

이전 노트북은 **SimpleX의 최고 Epoch가 실험 끝(70)**이었는데 160 Epoch까지만 늘렸습니다. 이번에는 **최대 700~900 Epoch를 탐색**하도록 개선했습니다. UltraGCN은 이전에 40~50 Epoch 부근에서 최고 성능을 내고 성능이 내려갔으므로 **최대 300~450 Epoch + 조기 종료**로 설계합니다. 다른 모델과 Epoch 수가 같아야 공정한 것은 아닙니다(모델별 Epoch당 Optimizer step 수가 다름).

- **BALANCED (기본): UltraGCN 5설정 + SimpleX 5설정 = 신규 10회**, 기존 LightGCN·NGCF·SGL·NCL 4회(기존 검증 예측을 별도 연결하지 않은 경우) + 약 30개 Rank Fusion 조합.
- **EXTENDED: 신규 16회 (8+8)**. 필요할 때만 실행.
- **QUICK: 신규 6회 (3+3)**. 동작 확인용.
- 모든 단일 모델은 같은 `val_train/val_truth/val_users`와 **pooled Recall@20** 평가, 학습에 Validation 정답 미사용.
- **Submission 생성 없음**. 점수가 좋은 단일·앙상블 후보만 추천하며 최종 제출은 별도 단계.
- 실험마다 최고 검증 점수의 가중치를 복원하여 Validation Top-20 출력. 동일 설정의 완료 파일은 설정/소스 해시 및 출력 유효성을 확인해 재사용.

**중요**: 이번 UltraGCN/SimpleX는 이전 노트북의 *논문 아이디어 기반 경량 구현*을 **그대로 사용**하며 정식 논문 코드와 동일하지 않습니다. 기본 앙상블은 각 모델의 **Top-20만 결합**하므로 Top-100 후보 결합과 다릅니다. 같은 Validation에서 반복적으로 튜닝하면 과적합될 수 있습니다.


## 1. Kaggle 환경 / 공식 대회 train·test 찾기

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, zipfile, time, os, csv, random
import pandas as pd
import torch
from IPython.display import display

WORK=Path('/kaggle/working')
INPUT=Path('/kaggle/input')
WORK.mkdir(parents=True,exist_ok=True)
OUT=WORK/'ultra_simplex_medium'
OUT.mkdir(parents=True,exist_ok=True)
VALID=WORK/'validation'
SPLIT_ID='f0dfad2f4431a876'
SEED=42
DEVICE='cuda:0'
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 꺼졌습니다. Settings → Accelerator → GPU를 켜 주세요.')
roots=sorted({p.parent for p in INPUT.rglob('train.csv') if (p.parent/'test.csv').exists()})
if len(roots)!=1:
    raise RuntimeError(f'대회 원본 데이터 소스가 정확히 1개 필요합니다: {roots}')
DATA=roots[0]
TRAIN=DATA/'train.csv'
TEST=DATA/'test.csv'
print('GPU:',torch.cuda.get_device_name(0))
print('원본 데이터:',DATA)
print('결과 폴더:',OUT)


## 2. 기존 검증된 평가 코드와 6개 모델 Runner 저장

별도의 pip 설치가 필요 없도록 팀 공통 `teamkit.py`, LightGCN/NGCF, SGL/NCL, UltraGCN/SimpleX runner를 내장했습니다. CUDA GPU가 필요합니다.

In [ ]:
%%writefile /kaggle/working/teamkit.py
"""Shared, standard-library-only evaluation utilities for the Amazon Top-20 team.

All interactions are targets, regardless of rating. IDs are kept as strings so
CSV round-trips do not change them. The public CSV contract is user_id,item_id.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import os
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Iterable, Mapping, Sequence


TOP_K = 20
SPLIT_SEED = 42
HOLDOUT_FRACTION = 0.20


@contextmanager
def open_source(path: str | None, competition_zip: str | None, member: str):
    """Open a UTF-8 file, either directly or as a named competition ZIP member."""
    if (path is None) == (competition_zip is None):
        raise ValueError("Provide exactly one of a file path or --competition-zip")
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive:
            with archive.open(member) as raw:
                with io.TextIOWrapper(raw, encoding="utf-8-sig", newline="") as stream:
                    yield stream
    else:
        with open(path, "r", encoding="utf-8-sig", newline="") as stream:
            yield stream


def _reader(stream, required: Sequence[str]) -> csv.DictReader:
    reader = csv.DictReader(stream)
    if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
        raise ValueError(f"Required columns: {', '.join(required)}")
    return reader


def _clean_id(value: str | None, name: str) -> str:
    if value is None or value == "" or value != value.strip() or any(c.isspace() for c in value):
        raise ValueError(f"Invalid {name}: {value!r}")
    return value


def _stable_rank(seed: int, user_id: str, item_id: str) -> bytes:
    payload = f"{seed}\0{user_id}\0{item_id}".encode("utf-8")
    return hashlib.blake2b(payload, digest_size=16).digest()


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _source_sha256(path: str | None, competition_zip: str | None, member: str) -> str:
    digest = hashlib.sha256()
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive, archive.open(member) as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    else:
        with open(path, "rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def make_split(
    train_path: str | None,
    competition_zip: str | None,
    output_dir: str | Path,
    seed: int = SPLIT_SEED,
    fraction: float = HOLDOUT_FRACTION,
    overwrite: bool = False,
) -> dict:
    """Make a stable, per-user split on unique user-item pairs.

    Hold out ceil(fraction * interactions), at most 20, leaving at least one
    interaction in fit. The hash rank is independent of input row order.
    """
    if not 0 < fraction < 1:
        raise ValueError("fraction must be between 0 and 1")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    names = ("val_train.csv", "val_truth.csv", "val_users.csv", "manifest.json")
    existing = [name for name in names if (output_dir / name).exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Existing split files: {existing}; use --overwrite to replace")

    with open_source(train_path, competition_zip, "train.csv") as stream:
        reader = _reader(stream, ("user_id", "item_id"))
        columns = list(reader.fieldnames or [])
        rows: list[tuple[str, ...]] = []
        user_items: dict[str, list[str]] = defaultdict(list)
        pairs: set[tuple[str, str]] = set()
        for row in reader:
            if None in row:
                raise ValueError("Malformed train.csv row with extra fields")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            pair = (user_id, item_id)
            if pair in pairs:
                raise ValueError(f"Duplicate interaction: {pair}")
            pairs.add(pair)
            user_items[user_id].append(item_id)
            rows.append(tuple(row[column] for column in columns))

    held: set[tuple[str, str]] = set()
    for user_id, items in user_items.items():
        if len(items) < 2:
            continue
        n_held = min(TOP_K, len(items) - 1, max(1, math.ceil(fraction * len(items))))
        chosen = sorted(items, key=lambda item: (_stable_rank(seed, user_id, item), item))[:n_held]
        held.update((user_id, item) for item in chosen)

    fit_path = output_dir / "val_train.csv"
    truth_path = output_dir / "val_truth.csv"
    users_path = output_dir / "val_users.csv"
    user_col, item_col = columns.index("user_id"), columns.index("item_id")
    with fit_path.open("w", encoding="utf-8", newline="") as fit_stream, truth_path.open(
        "w", encoding="utf-8", newline=""
    ) as truth_stream:
        fit_writer, truth_writer = csv.writer(fit_stream), csv.writer(truth_stream)
        fit_writer.writerow(columns)
        truth_writer.writerow(("user_id", "item_id"))
        for row in rows:
            pair = (row[user_col], row[item_col])
            if pair in held:
                truth_writer.writerow(pair)
            else:
                fit_writer.writerow(row)

    with users_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id",))
        for user_id in user_items:
            if len(user_items[user_id]) >= 2:
                writer.writerow((user_id,))

    source_hash = _source_sha256(train_path, competition_zip, "train.csv")
    split_key = f"{source_hash}|blake2b16|{seed}|{fraction}|ceil20keep1"
    manifest = {
        "split_id": hashlib.sha256(split_key.encode()).hexdigest()[:16],
        "algorithm": "per-user blake2b-128 rank over seed, user_id, item_id",
        "seed": seed,
        "holdout_fraction": fraction,
        "holdout_count": "min(20, n-1, max(1, ceil(fraction*n))) for n>=2",
        "source_train_sha256": source_hash,
        "source_rows": len(rows),
        "source_users": len(user_items),
        "fit_rows": len(rows) - len(held),
        "truth_rows": len(held),
        "validation_users": sum(len(items) >= 2 for items in user_items.values()),
        "files_sha256": {name: _sha256(output_dir / name) for name in names[:3]},
    }
    with (output_dir / "manifest.json").open("w", encoding="utf-8") as stream:
        json.dump(manifest, stream, ensure_ascii=False, indent=2)
        stream.write("\n")
    return manifest


def verify_split(output_dir: str | Path) -> dict:
    output_dir = Path(output_dir)
    with (output_dir / "manifest.json").open("r", encoding="utf-8") as stream:
        manifest = json.load(stream)
    for name, expected in manifest["files_sha256"].items():
        actual = _sha256(output_dir / name)
        if actual != expected:
            raise ValueError(f"Split file checksum mismatch: {name}")
    return manifest


def read_users(path: str | Path) -> list[str]:
    users: list[str] = []
    seen: set[str] = set()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id",)):
            if None in row:
                raise ValueError(f"Malformed user CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in seen:
                raise ValueError(f"Duplicate user in {path}: {user_id}")
            seen.add(user_id)
            users.append(user_id)
    return users


def read_interactions(path: str | Path) -> tuple[dict[str, set[str]], Counter[str]]:
    """Return seen items by user and item popularity from a fit CSV."""
    seen: dict[str, set[str]] = defaultdict(set)
    popularity: Counter[str] = Counter()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed interaction CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id not in seen[user_id]:
                seen[user_id].add(item_id)
                popularity[item_id] += 1
    return dict(seen), popularity


def read_truth(path: str | Path) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed truth CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id in truth[user_id]:
                raise ValueError(f"Duplicate truth item for user {user_id}: {item_id}")
            truth[user_id].add(item_id)
    return dict(truth)


def read_predictions(path: str | Path) -> dict[str, list[str]]:
    predictions: dict[str, list[str]] = {}
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["user_id", "item_id"]:
            raise ValueError("Prediction columns must be exactly user_id,item_id")
        for row in reader:
            if None in row:
                raise ValueError("Malformed prediction CSV row")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in predictions:
                raise ValueError(f"Duplicate prediction user: {user_id}")
            value = row["item_id"]
            if value is None:
                raise ValueError(f"Missing item list for user {user_id}")
            items = value.split(" ")
            if len(items) != TOP_K or any(not item or item != item.strip() for item in items):
                raise ValueError(f"User {user_id} needs exactly {TOP_K} space-separated item IDs")
            if len(set(items)) != TOP_K:
                raise ValueError(f"Duplicate recommended item for user {user_id}")
            predictions[user_id] = items
    return predictions


def validate_predictions(
    predictions: Mapping[str, Sequence[str]],
    expected_users: Iterable[str],
    seen: Mapping[str, set[str]],
    catalog: set[str],
) -> None:
    expected = set(expected_users)
    actual = set(predictions)
    if actual != expected:
        missing, extra = expected - actual, actual - expected
        raise ValueError(f"User mismatch: {len(missing)} missing, {len(extra)} extra")
    for user_id, items in predictions.items():
        if len(items) != TOP_K or len(set(items)) != TOP_K:
            raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
        unknown = set(items) - catalog
        if unknown:
            raise ValueError(f"Unknown item for user {user_id}: {sorted(unknown)[:3]}")
        repeated = set(items) & seen.get(user_id, set())
        if repeated:
            raise ValueError(f"Seen item recommended to user {user_id}: {sorted(repeated)[:3]}")


def recall_at_20(predictions: Mapping[str, Sequence[str]], truth: Mapping[str, set[str]]) -> dict:
    """Competition-style pooled Recall@20, not mean of user-level recalls."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user_id]) for user_id, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if denominator == 0:
        raise ValueError("No validation truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def top20_unseen(
    user_id: str,
    ranked_candidates: Iterable[str],
    seen: Mapping[str, set[str]],
    fallback_items: Iterable[str],
    catalog: set[str],
) -> list[str]:
    """Filter a model ranking; fill short lists with popular unseen items."""
    chosen: list[str] = []
    used = set(seen.get(user_id, set()))
    for source in (ranked_candidates, fallback_items):
        for item_id in source:
            item_id = str(item_id)
            if item_id not in catalog or item_id in used:
                continue
            chosen.append(item_id)
            used.add(item_id)
            if len(chosen) == TOP_K:
                return chosen
    raise ValueError(f"Fewer than {TOP_K} available unseen items for user {user_id}")


def write_predictions(path: str | Path, users: Iterable[str], predictions: Mapping[str, Sequence[str]]) -> None:
    """Write one row per user in the provided order, using single spaces."""
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user_id in users:
            items = predictions[user_id]
            if len(items) != TOP_K or len(set(items)) != TOP_K:
                raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
            writer.writerow((user_id, " ".join(map(str, items))))


def filter_reviews(
    fit_path: str | Path,
    output_path: str | Path,
    reviews_path: str | None = None,
    competition_zip: str | None = None,
) -> dict:
    """Keep only training-pair reviews to prevent validation target leakage."""
    seen, _ = read_interactions(fit_path)
    allowed = {(user_id, item_id) for user_id, items in seen.items() for item_id in items}
    found: set[tuple[str, str]] = set()
    written = 0
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_path = tempfile.mkstemp(prefix=".reviews-", suffix=".tmp", dir=output_path.parent)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="") as output, open_source(
            reviews_path, competition_zip, "train_reviews.jsonl"
        ) as source:
            for line in source:
                review = json.loads(line)
                pair = (str(review["user_id"]), str(review["item_id"]))
                if pair in allowed:
                    output.write(line)
                    found.add(pair)
                    written += 1
        if found != allowed:
            raise ValueError(f"Review coverage mismatch: {len(allowed - found)} fit pairs have no review")
        os.replace(temporary_path, output_path)
    finally:
        if os.path.exists(temporary_path):
            os.unlink(temporary_path)
    return {"written_reviews": written, "distinct_fit_pairs": len(allowed)}


def _print_json(value: dict) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def main(argv: Sequence[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Shared Amazon Top-20 evaluation tools")
    commands = parser.add_subparsers(dest="command", required=True)

    split = commands.add_parser("split", help="Create deterministic validation CSVs")
    source = split.add_mutually_exclusive_group(required=True)
    source.add_argument("--train", help="Path to extracted train.csv")
    source.add_argument("--competition-zip", help="Path to original competition ZIP")
    split.add_argument("--out", required=True, help="Output directory")
    split.add_argument("--seed", type=int, default=SPLIT_SEED)
    split.add_argument("--fraction", type=float, default=HOLDOUT_FRACTION)
    split.add_argument("--overwrite", action="store_true")

    verify = commands.add_parser("verify-split", help="Check shared split file hashes")
    verify.add_argument("--dir", required=True)

    score = commands.add_parser("score", help="Validate and score validation predictions")
    score.add_argument("--pred", required=True)
    score.add_argument("--fit", required=True)
    score.add_argument("--truth", required=True)

    check = commands.add_parser("check", help="Validate a final test submission")
    check.add_argument("--pred", required=True)
    check.add_argument("--train", required=True)
    check.add_argument("--test", required=True)

    reviews = commands.add_parser("filter-reviews", help="Remove held-out reviews")
    reviews.add_argument("--fit", required=True)
    review_source = reviews.add_mutually_exclusive_group(required=True)
    review_source.add_argument("--reviews", help="Path to extracted train_reviews.jsonl")
    review_source.add_argument("--competition-zip", help="Path to original competition ZIP")
    reviews.add_argument("--out", required=True)

    args = parser.parse_args(argv)
    if args.command == "split":
        _print_json(make_split(args.train, args.competition_zip, args.out, args.seed, args.fraction, args.overwrite))
    elif args.command == "verify-split":
        manifest = verify_split(args.dir)
        _print_json({"split_id": manifest["split_id"], "verified": True})
    elif args.command == "score":
        seen, popularity = read_interactions(args.fit)
        truth = read_truth(args.truth)
        predictions = read_predictions(args.pred)
        catalog = set(popularity) | {item for items in truth.values() for item in items}
        validate_predictions(predictions, truth, seen, catalog)
        result = recall_at_20(predictions, truth)
        result["valid_submission_format"] = True
        _print_json(result)
    elif args.command == "check":
        seen, popularity = read_interactions(args.train)
        users = read_users(args.test)
        predictions = read_predictions(args.pred)
        validate_predictions(predictions, users, seen, set(popularity))
        _print_json({"valid": True, "users": len(users), "items_per_user": TOP_K})
    elif args.command == "filter-reviews":
        _print_json(filter_reviews(args.fit, args.out, args.reviews, args.competition_zip))
    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except (ValueError, FileExistsError, FileNotFoundError, KeyError, zipfile.BadZipFile) as error:
        print(f"ERROR: {error}", file=sys.stderr)
        sys.exit(1)

In [ ]:
%%writefile /kaggle/working/graph_models.py
"""Single-GPU LightGCN / NGCF experiments for the shared Amazon Top-20 split.

Example:
    python graph_models.py --model lightgcn \
      --fit ../amazon_topk_teamkit/validation/val_train.csv \
      --users ../amazon_topk_teamkit/validation/val_users.csv \
      --truth ../amazon_topk_teamkit/validation/val_truth.csv \
      --out lightgcn_val.csv

No metadata or reviews are read. Train interactions are the only graph edges.
"""

from __future__ import annotations

import argparse
import copy
import csv
import json
import math
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ModuleNotFoundError:
    torch = None
    nn = None
    F = None


TOP_K = 20
_BaseModel = nn.Module if nn is not None else object


def _id_key(value: str):
    return (0, int(value)) if value.isdecimal() else (1, value)


def _csv_rows(path: str | Path, required: tuple[str, ...]):
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
            raise ValueError(f"{path} must contain {', '.join(required)}")
        for row in reader:
            if None in row:
                raise ValueError(f"Malformed CSV row in {path}")
            yield row


def _identifier(value: str | None, column: str) -> str:
    if not value or value != value.strip() or any(char.isspace() for char in value):
        raise ValueError(f"Invalid {column}: {value!r}")
    return value


@dataclass
class InteractionData:
    user_ids: list[str]
    item_ids: list[str]
    user_index: dict[str, int]
    item_index: dict[str, int]
    positive_users: list[int]
    positive_items: list[int]
    seen_items: dict[int, set[int]]
    target_users: list[str]

    @property
    def num_users(self) -> int:
        return len(self.user_ids)

    @property
    def num_items(self) -> int:
        return len(self.item_ids)

    @property
    def num_edges(self) -> int:
        return len(self.positive_users)


def load_data(fit_path: str | Path, users_path: str | Path) -> InteractionData:
    """Index unique positive edges; do not filter interactions by rating."""
    edges: set[tuple[str, str]] = set()
    for row in _csv_rows(fit_path, ("user_id", "item_id")):
        edges.add((_identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")))
    if not edges:
        raise ValueError("Empty fit interactions")

    target_users: list[str] = []
    target_set: set[str] = set()
    for row in _csv_rows(users_path, ("user_id",)):
        user_id = _identifier(row["user_id"], "user_id")
        if user_id in target_set:
            raise ValueError(f"Duplicate target user: {user_id}")
        target_set.add(user_id)
        target_users.append(user_id)

    train_users = {user for user, _ in edges}
    unseen_users = target_set - train_users
    if unseen_users:
        raise ValueError(f"{len(unseen_users)} target users have no training interaction")
    user_ids = sorted(train_users, key=_id_key)
    item_ids = sorted({item for _, item in edges}, key=_id_key)
    if len(item_ids) <= TOP_K:
        raise ValueError("Need more than 20 known items to recommend 20 unseen items")
    user_index = {value: index for index, value in enumerate(user_ids)}
    item_index = {value: index for index, value in enumerate(item_ids)}
    positive_users: list[int] = []
    positive_items: list[int] = []
    seen_items: dict[int, set[int]] = defaultdict(set)
    for user, item in sorted(edges, key=lambda pair: (_id_key(pair[0]), _id_key(pair[1]))):
        u, i = user_index[user], item_index[item]
        positive_users.append(u)
        positive_items.append(i)
        seen_items[u].add(i)
    for user in target_users:
        if len(seen_items[user_index[user]]) + TOP_K > len(item_ids):
            raise ValueError(f"User {user} has fewer than 20 unseen items")
    return InteractionData(
        user_ids, item_ids, user_index, item_index, positive_users,
        positive_items, dict(seen_items), target_users,
    )


def load_truth(path: str | Path, target_users: Sequence[str]) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    for row in _csv_rows(path, ("user_id", "item_id")):
        user, item = _identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")
        if item in truth[user]:
            raise ValueError(f"Duplicate truth pair: {user}, {item}")
        truth[user].add(item)
    if set(truth) != set(target_users):
        raise ValueError("Truth and target user sets differ")
    return dict(truth)


def recall_at_20(predictions: dict[str, list[str]], truth: dict[str, set[str]]) -> dict:
    """Pooled competition Recall@20: total hits / sum(min(20, |GT_u|))."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user]) for user, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if not denominator:
        raise ValueError("No truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def normalized_graph(data: InteractionData, device):
    """D^-1/2 A D^-1/2 for the undirected user-item graph."""
    if torch is None:
        raise RuntimeError("PyTorch is required to build the graph")
    u = torch.tensor(data.positive_users, dtype=torch.long)
    i = torch.tensor(data.positive_items, dtype=torch.long) + data.num_users
    rows = torch.cat((u, i))
    cols = torch.cat((i, u))
    degree = torch.bincount(rows, minlength=data.num_users + data.num_items).float()
    values = (degree[rows] * degree[cols]).rsqrt()
    indices = torch.stack((rows, cols)).to(device)
    return torch.sparse_coo_tensor(
        indices, values.to(device),
        (data.num_users + data.num_items, data.num_users + data.num_items),
        device=device,
    ).coalesce()


def sparse_edge_dropout(graph, probability: float):
    if probability <= 0:
        return graph
    if not 0 <= probability < 1:
        raise ValueError("Edge dropout must be in [0, 1)")
    keep = torch.rand(graph._nnz(), device=graph.device) >= probability
    return torch.sparse_coo_tensor(
        graph.indices()[:, keep], graph.values()[keep] / (1 - probability),
        graph.shape, device=graph.device,
    ).coalesce()


class LightGCN(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int):
        if nn is None:
            raise RuntimeError("PyTorch is required to train LightGCN")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.layers = layers
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for _ in range(self.layers):
            current = torch.sparse.mm(graph, current)
            layer_outputs.append(current)
        final = torch.stack(layer_outputs, dim=0).mean(dim=0)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


class NGCF(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int, message_dropout: float):
        if nn is None:
            raise RuntimeError("PyTorch is required to train NGCF")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.sum_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.bi_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.message_dropout = message_dropout
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
        for layer in (*self.sum_linears, *self.bi_linears):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for sum_linear, bi_linear in zip(self.sum_linears, self.bi_linears):
            neighbors = torch.sparse.mm(graph, current)
            current = F.leaky_relu(sum_linear(current + neighbors) + bi_linear(current * neighbors), negative_slope=0.2)
            current = F.dropout(current, p=self.message_dropout, training=self.training)
            current = F.normalize(current, p=2, dim=1)
            layer_outputs.append(current)
        final = torch.cat(layer_outputs, dim=1)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


def sample_triples(data: InteractionData, count: int, rng: random.Random, device):
    """Sample one uniformly random unseen item per sampled positive edge."""
    if count <= 0:
        raise ValueError("samples-per-epoch must be positive")
    edge_indices = [rng.randrange(data.num_edges) for _ in range(count)]
    users = [data.positive_users[index] for index in edge_indices]
    positive = [data.positive_items[index] for index in edge_indices]
    negative = [rng.randrange(data.num_items) for _ in range(count)]
    for position, (user, item) in enumerate(zip(users, negative)):
        if item in data.seen_items[user]:
            while True:
                candidate = rng.randrange(data.num_items)
                if candidate not in data.seen_items[user]:
                    negative[position] = candidate
                    break
    return (
        torch.tensor(users, dtype=torch.long, device=device),
        torch.tensor(positive, dtype=torch.long, device=device),
        torch.tensor(negative, dtype=torch.long, device=device),
    )


@torch.no_grad() if torch is not None else (lambda function: function)
def predict_top20(model, graph, data: InteractionData, user_batch_size: int) -> dict[str, list[str]]:
    """Score all known items in bounded user batches and mask fit interactions."""
    if user_batch_size <= 0:
        raise ValueError("eval-user-batch-size must be positive")
    model.eval()
    user_vectors, item_vectors = model(graph)
    item_matrix = item_vectors.T.contiguous()
    predictions: dict[str, list[str]] = {}
    for start in range(0, len(data.target_users), user_batch_size):
        batch = data.target_users[start : start + user_batch_size]
        user_indices = torch.tensor([data.user_index[user] for user in batch], dtype=torch.long, device=graph.device)
        scores = user_vectors[user_indices] @ item_matrix
        masked_rows: list[int] = []
        masked_columns: list[int] = []
        for local_row, user_index in enumerate(user_indices.tolist()):
            for item_index in data.seen_items[user_index]:
                masked_rows.append(local_row)
                masked_columns.append(item_index)
        if masked_rows:
            row_index = torch.tensor(masked_rows, dtype=torch.long, device=graph.device)
            column_index = torch.tensor(masked_columns, dtype=torch.long, device=graph.device)
            scores[row_index, column_index] = -float("inf")
        top = torch.topk(scores, k=TOP_K, dim=1).indices.cpu().tolist()
        for user_id, item_indices in zip(batch, top):
            predictions[user_id] = [data.item_ids[index] for index in item_indices]
    return predictions


def write_predictions(path: str | Path, users: Sequence[str], predictions: dict[str, list[str]]) -> None:
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user in users:
            writer.writerow((user, " ".join(predictions[user])))


def _device(name: str):
    if torch is None:
        raise RuntimeError("PyTorch is not installed. Use Kaggle GPU Notebook or install torch.")
    if name == "auto":
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if name.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    return torch.device(name)


def _training_loss(model, user_vectors, item_vectors, sampled, regularization: float):
    users, positives, negatives = sampled
    positive_scores = (user_vectors[users] * item_vectors[positives]).sum(dim=1)
    negative_scores = (user_vectors[users] * item_vectors[negatives]).sum(dim=1)
    bpr = F.softplus(negative_scores - positive_scores).mean()
    raw_u = model.user_embedding(users)
    raw_p = model.item_embedding(positives)
    raw_n = model.item_embedding(negatives)
    reg = (raw_u.square().sum(dim=1) + raw_p.square().sum(dim=1) + raw_n.square().sum(dim=1)).mean() / 2
    return bpr + regularization * reg, bpr, reg


def run(args: argparse.Namespace) -> dict:
    device = _device(args.device)
    if args.layers < 1 or args.embedding_dim < 1 or args.epochs < 1:
        raise ValueError("layers, embedding-dim and epochs must be positive")
    if not 0 <= args.edge_dropout < 1 or not 0 <= args.message_dropout < 1:
        raise ValueError("Dropout probabilities must be in [0, 1)")
    if args.reg < 0 or args.lr <= 0:
        raise ValueError("reg must be nonnegative and lr positive")
    torch.manual_seed(args.seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(args.seed)
    rng = random.Random(args.seed)
    data = load_data(args.fit, args.users)
    truth = load_truth(args.truth, data.target_users) if args.truth else None
    graph = normalized_graph(data, device)
    if args.model == "lightgcn":
        model = LightGCN(data.num_users, data.num_items, args.embedding_dim, args.layers)
    else:
        model = NGCF(data.num_users, data.num_items, args.embedding_dim, args.layers, args.message_dropout)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    print(json.dumps({
        "model": args.model, "device": str(device), "users": data.num_users,
        "items": data.num_items, "fit_edges": data.num_edges,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
    }), flush=True)

    best_score = -math.inf
    best_state = None
    best_epoch = None
    stale_evaluations = 0
    last_loss = None
    trained_epochs = 0
    for epoch in range(1, args.epochs + 1):
        trained_epochs = epoch
        start = time.monotonic()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sampled = sample_triples(data, args.samples_per_epoch, rng, device)
        epoch_graph = sparse_edge_dropout(graph, args.edge_dropout) if args.model == "ngcf" else graph
        user_vectors, item_vectors = model(epoch_graph)
        loss, bpr, reg = _training_loss(model, user_vectors, item_vectors, sampled, args.reg)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
        record = {
            "epoch": epoch, "loss": round(last_loss, 6),
            "bpr": round(float(bpr.detach().cpu()), 6),
            "regularization": round(float(reg.detach().cpu()), 6),
            "seconds": round(time.monotonic() - start, 2),
        }
        del sampled, user_vectors, item_vectors, loss, bpr, reg
        should_evaluate = truth is not None and (epoch == 1 or epoch % args.eval_every == 0 or epoch == args.epochs)
        if should_evaluate:
            predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
            metric = recall_at_20(predictions, truth)
            record.update(metric)
            if metric["recall_at_20"] > best_score:
                best_score = metric["recall_at_20"]
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                stale_evaluations = 0
                record["best"] = True
            else:
                stale_evaluations += 1
                record["best"] = False
        print(json.dumps(record), flush=True)
        if should_evaluate and args.patience > 0 and stale_evaluations >= args.patience:
            print(json.dumps({"early_stop_epoch": epoch}), flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
    write_predictions(args.out, data.target_users, predictions)
    result = {
        "model": args.model, "device": str(device), "fit_edges": data.num_edges,
        "users": len(data.target_users), "output": str(args.out), "last_loss": last_loss,
        "trained_epochs": trained_epochs, "best_epoch": best_epoch,
        "epochs_requested": args.epochs, "seed": args.seed,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
        "learning_rate": args.lr, "regularization": args.reg,
        "edge_dropout": args.edge_dropout if args.model == "ngcf" else 0.0,
        "message_dropout": args.message_dropout if args.model == "ngcf" else 0.0,
    }
    if truth is not None:
        result.update(recall_at_20(predictions, truth))
    print(json.dumps({"final": result}), flush=True)
    return result


def main(argv: Sequence[str] | None = None) -> dict:
    parser = argparse.ArgumentParser(description="LightGCN / NGCF on the shared Amazon Top-20 split")
    parser.add_argument("--model", choices=("lightgcn", "ngcf"), required=True)
    parser.add_argument("--fit", required=True, help="val_train.csv or full train.csv")
    parser.add_argument("--users", required=True, help="val_users.csv or test.csv")
    parser.add_argument("--truth", help="val_truth.csv; omit for final test predictions")
    parser.add_argument("--out", required=True, help="Prediction CSV")
    parser.add_argument("--metrics-out", help="Optional JSON file for run settings and results")
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--embedding-dim", type=int, default=64)
    parser.add_argument("--layers", type=int, default=3)
    parser.add_argument("--samples-per-epoch", type=int, default=100000)
    parser.add_argument("--eval-user-batch-size", type=int, default=256)
    parser.add_argument("--eval-every", type=int, default=5)
    parser.add_argument("--patience", type=int, default=4, help="Validation evaluations without improvement; 0 disables")
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--reg", type=float, default=0.0001)
    parser.add_argument("--edge-dropout", type=float, default=0.0, help="Optional NGCF edge dropout")
    parser.add_argument("--message-dropout", type=float, default=0.1, help="NGCF only")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto", help="auto, cpu, cuda:0")
    args = parser.parse_args(argv)
    if args.eval_every < 1 or args.patience < 0:
        parser.error("eval-every must be positive and patience nonnegative")
    result = run(args)
    if args.metrics_out:
        Path(args.metrics_out).write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/sgl_ncl_runner.py
"""LightGCN backbone with SGL-ED or NCL-style SSL, for the existing team kit.

Faithfulness note:
- SGL-ED: independently edge-subsampled, re-normalized graph views, InfoNCE.
- NCL: layer-0 versus 2h-hop structural InfoNCE, user/item prototypes from
  detached spherical torch k-means; not bit-for-bit identical to RecBole/FAISS.
Validation truth is used strictly for model selection and reporting.
"""
from __future__ import annotations
import argparse
import copy
import json
import math
import random
import time
from pathlib import Path

import torch
import torch.nn.functional as F

from graph_models import (
    LightGCN, _training_loss, load_data, load_truth, normalized_graph,
    predict_top20, recall_at_20, sample_triples, write_predictions,
)


def edge_view(positive_users, positive_items, n_users, n_items, drop, device):
    """Independent sampled bipartite graph, symmetric, degree re-normalized."""
    if not 0 <= drop < 1:
        raise ValueError('drop ratio must be in [0,1)')
    keep = torch.rand(positive_users.numel(), device=device) >= drop
    u, i = positive_users[keep], positive_items[keep] + n_users
    row = torch.cat((u, i))
    col = torch.cat((i, u))
    counts = torch.bincount(row, minlength=n_users + n_items).float()
    values = counts[row].clamp_min(1).rsqrt() * counts[col].clamp_min(1).rsqrt()
    return torch.sparse_coo_tensor(
        torch.stack((row, col)), values,
        (n_users + n_items, n_users + n_items), device=device
    ).coalesce()


def propagation(model, graph, depth):
    curr = torch.cat((model.user_embedding.weight, model.item_embedding.weight), dim=0)
    steps = [curr]
    for _ in range(depth):
        curr = torch.sparse.mm(graph, curr)
        steps.append(curr)
    mean = torch.stack(steps[:model.layers + 1], 0).mean(0)
    n_users = model.user_embedding.num_embeddings
    return mean[:n_users], mean[n_users:], steps


def nce_full(anchor, positive, candidate_all, tau):
    """Sampled anchor nodes, ALL same-type nodes used in InfoNCE denominator."""
    anchor = F.normalize(anchor, dim=-1)
    positive = F.normalize(positive, dim=-1)
    candidate_all = F.normalize(candidate_all, dim=-1)
    logits = anchor @ candidate_all.T / tau
    pos = (anchor * positive).sum(-1) / tau
    return (torch.logsumexp(logits, dim=1) - pos).mean()


def sgl_info_nce(view1, view2, sampled_users, sampled_items, n_users, tau):
    loss_u = nce_full(
        view1[0][sampled_users], view2[0][sampled_users], view2[0], tau
    )
    loss_i = nce_full(
        view1[1][sampled_items], view2[1][sampled_items], view2[1], tau
    )
    return loss_u + loss_i


@torch.no_grad()
def spherical_kmeans(x, n_clusters, iterations=5):
    """GPU spherical k-means to avoid requiring faiss-gpu/RecBole."""
    x = F.normalize(x.detach(), dim=1)
    n = x.size(0)
    if n_clusters > n:
        raise ValueError('Clusters exceed node count')
    centers = x[torch.randperm(n, device=x.device)[:n_clusters]].clone()
    for _ in range(iterations):
        labels = (x @ centers.T).argmax(dim=1)
        sums = torch.zeros_like(centers)
        sums.index_add_(0, labels, x)
        counts = torch.bincount(labels, minlength=n_clusters)
        empty = torch.where(counts == 0)[0]
        if empty.numel():
            sums[empty] = x[torch.randint(n, (empty.numel(),), device=x.device)]
        centers = F.normalize(sums, dim=1)
    labels = (x @ centers.T).argmax(dim=1)
    return centers.detach(), labels.detach()


def prototype_nce(raw_users, raw_items, sample_users, sample_items,
                  user_clusters, item_clusters, tau):
    cent_u, lab_u = user_clusters
    cent_i, lab_i = item_clusters
    u = F.normalize(raw_users[sample_users], dim=-1)
    i = F.normalize(raw_items[sample_items], dim=-1)
    lu = F.cross_entropy(u @ cent_u.T / tau, lab_u[sample_users])
    li = F.cross_entropy(i @ cent_i.T / tau, lab_i[sample_items])
    return lu + li


def evaluate(model, graph, data, truth, eval_batch):
    predictions = predict_top20(model, graph, data, eval_batch)
    return recall_at_20(predictions, truth)


def train_run(a):
    if a.epochs < 1 or a.layers < 1 or a.dim < 1 or a.samples_per_epoch < 1:
        raise ValueError('epochs/layers/dim/samples_per_epoch must be positive')
    if a.ssl_nodes < 1 or a.eval_every < 1 or a.tau <= 0 or a.lr <= 0:
        raise ValueError('ssl_nodes/eval_every/tau/lr must be positive')
    if a.model == 'ncl' and (a.hyper_layers < 1 or a.clusters < 2 or a.cluster_every < 1):
        raise ValueError('invalid NCL clustering/hyper-layer configuration')
    if a.ssl_weight < 0 or a.proto_weight < 0 or a.reg < 0:
        raise ValueError('loss weights must be nonnegative')
    if a.device.startswith('cuda') and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but not available')
    device = torch.device(a.device)
    random.seed(a.seed)
    torch.manual_seed(a.seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(a.seed)
    rng = random.Random(a.seed)
    data = load_data(a.fit, a.users)
    truth = load_truth(a.truth, data.target_users) if a.truth else None
    graph = normalized_graph(data, device)
    edge_u = torch.tensor(data.positive_users, dtype=torch.long, device=device)
    edge_i = torch.tensor(data.positive_items, dtype=torch.long, device=device)
    model = LightGCN(data.num_users, data.num_items, a.dim, a.layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=a.lr)
    print(json.dumps({'info':'START', 'model':a.model, 'users':data.num_users, 'items':data.num_items,
                      'fit_edges':data.num_edges,'layers':a.layers,'dim':a.dim,
                      'device':str(device),'epochs':a.epochs, 'ssl_weight':a.ssl_weight,
                      'proto_weight':a.proto_weight}), flush=True)
    best_score = -1.0
    best_epoch = None
    best_state = None
    stale_evals = 0
    current_clusters = None
    last_loss = None
    trained_epochs = 0
    out_path = Path(a.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    for epoch in range(1, a.epochs + 1):
        tick = time.monotonic()
        trained_epochs = epoch
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sampled = sample_triples(data, a.samples_per_epoch, rng, device)
        anchor_u = torch.unique(sampled[0][:a.ssl_nodes])
        anchor_i = torch.unique(sampled[1][:a.ssl_nodes])
        if a.model == 'ncl':
            u_vec, i_vec, steps = propagation(model, graph, max(a.layers, 2 * a.hyper_layers))
        else:
            u_vec, i_vec = model(graph)
            steps = None
        bpr_reg, bpr, reg = _training_loss(model, u_vec, i_vec, sampled, a.reg)
        ssl = bpr_reg.new_zeros(())
        proto = bpr_reg.new_zeros(())
        if a.model == 'sgl' and a.ssl_weight > 0:
            g1 = edge_view(edge_u, edge_i, data.num_users, data.num_items, a.drop, device)
            g2 = edge_view(edge_u, edge_i, data.num_users, data.num_items, a.drop, device)
            e1 = model(g1)
            e2 = model(g2)
            ssl = sgl_info_nce(e1, e2, anchor_u, anchor_i, data.num_users, a.tau)
        elif a.model == 'ncl':
            raw = steps[0]
            context = steps[2 * a.hyper_layers]
            raw_user = raw[:data.num_users]
            raw_item = raw[data.num_users:]
            ctx_user = context[:data.num_users]
            ctx_item = context[data.num_users:]
            if a.ssl_weight > 0:
                ssl = (nce_full(ctx_user[anchor_u], raw_user[anchor_u], raw_user, a.tau)
                       + a.alpha * nce_full(ctx_item[anchor_i], raw_item[anchor_i], raw_item, a.tau))
            if a.proto_weight > 0 and epoch >= a.warmup:
                if current_clusters is None or (epoch - a.warmup) % a.cluster_every == 0:
                    current_clusters = (
                        spherical_kmeans(raw_user, a.clusters),
                        spherical_kmeans(raw_item, a.clusters),
                    )
                proto = prototype_nce(raw_user, raw_item, anchor_u, anchor_i,
                                      current_clusters[0], current_clusters[1], a.tau)
        loss = bpr_reg + a.ssl_weight * ssl + a.proto_weight * proto
        if not torch.isfinite(loss):
            raise RuntimeError(f'Loss non-finite at epoch {epoch}: {float(loss)}')
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach())
        record = {'epoch':epoch, 'loss':round(last_loss,6),
                  'bpr':round(float(bpr.detach()),6),
                  'reg':round(float(reg.detach()),6),
                  'ssl':round(float(ssl.detach()),6),
                  'proto':round(float(proto.detach()),6),
                  'seconds':round(time.monotonic()-tick,3)}
        del sampled, u_vec, i_vec, steps, loss, bpr_reg, bpr, reg, ssl, proto
        should_eval = truth is not None and (epoch == 1 or epoch % a.eval_every == 0 or epoch == a.epochs)
        if should_eval:
            score = evaluate(model, graph, data, truth, a.eval_batch)
            record.update(score)
            if score['recall_at_20'] > best_score:
                best_score = score['recall_at_20']
                best_epoch = epoch
                best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                stale_evals = 0
                record['best'] = True
            else:
                stale_evals += 1
                record['best'] = False
        elif epoch % 50 != 0:
            continue  # emit concise logs only, not 700+ lines per run
        print(json.dumps(record), flush=True)
        if should_eval and a.patience > 0 and stale_evals >= a.patience:
            print(json.dumps({'early_stop_epoch':epoch}),flush=True)
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    predictions = predict_top20(model, graph, data, a.eval_batch)
    write_predictions(out_path, data.target_users, predictions)
    result = {
        'model':a.model, 'output':str(out_path), 'best_epoch':best_epoch,
        'trained_epochs':trained_epochs,'epochs_requested':a.epochs,'device':str(device),
        'layers':a.layers,'dim':a.dim,'lr':a.lr,'reg':a.reg,'ssl_weight':a.ssl_weight,
        'proto_weight':a.proto_weight,'drop':a.drop,'tau':a.tau,'clusters':a.clusters,
        'warmup':a.warmup,'hyper_layers':a.hyper_layers,'alpha':a.alpha,
        'samples_per_epoch':a.samples_per_epoch,'seed':a.seed,'last_loss':last_loss,
    }
    if truth is not None:
        result.update(recall_at_20(predictions,truth))
    if a.metrics:
        Path(a.metrics).parent.mkdir(parents=True,exist_ok=True)
        Path(a.metrics).write_text(json.dumps(result,indent=2),encoding='utf-8')
    if a.checkpoint:
        torch.save({'model':a.model,'weights':model.state_dict(),'config':vars(a)},a.checkpoint)
    print(json.dumps({'final':result}),flush=True)
    return result


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--model',choices=['sgl','ncl'],required=True)
    p.add_argument('--fit',required=True)
    p.add_argument('--users',required=True)
    p.add_argument('--truth',default=None)
    p.add_argument('--out',required=True)
    p.add_argument('--metrics',default=None)
    p.add_argument('--checkpoint',default=None)
    p.add_argument('--device',default='cuda:0')
    p.add_argument('--epochs',type=int,default=700)
    p.add_argument('--eval-every',type=int,default=25)
    p.add_argument('--patience',type=int,default=0)
    p.add_argument('--eval-batch',type=int,default=256)
    p.add_argument('--dim',type=int,default=64)
    p.add_argument('--layers',type=int,default=3)
    p.add_argument('--lr',type=float,default=.002)
    p.add_argument('--reg',type=float,default=.0001)
    p.add_argument('--samples-per-epoch',type=int,default=100000)
    p.add_argument('--seed',type=int,default=42)
    p.add_argument('--ssl-nodes',type=int,default=192)
    p.add_argument('--ssl-weight',type=float,default=.05)
    p.add_argument('--drop',type=float,default=.1)
    p.add_argument('--tau',type=float,default=.2)
    p.add_argument('--proto-weight',type=float,default=.005)
    p.add_argument('--clusters',type=int,default=64)
    p.add_argument('--cluster-every',type=int,default=50)
    p.add_argument('--warmup',type=int,default=50)
    p.add_argument('--hyper-layers',type=int,default=1)
    p.add_argument('--alpha',type=float,default=1.)
    a=p.parse_args()
    train_run(a)

if __name__=='__main__':
    main()

In [ ]:
%%writefile /kaggle/working/ultra_simplex_runner.py
"""UltraGCN-style and SimpleX-style implicit top-k recommendation.

Designed for the team's provided split and single CUDA GPU (also CPU testable).
Faithfulness notes:
- UltraGCN-style: direct user/item embeddings, degree-aware weighted BCE,
  and item-item co-occurrence constraint. This is a streamlined approximation
  of UltraGCN, not a bit-for-bit reproduction of the published implementation.
- SimpleX-style: user ID + mean item-history tower, cosine contrastive loss,
  leave-one-out history for positive training samples. This is a compact
  implementation of the SimpleX family, not the official codebase.
- Validation positives must NEVER be included in fit history or item neighbors.
"""
from __future__ import annotations

import argparse
import copy
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import torch
from torch import nn
import torch.nn.functional as F

from graph_models import load_data, load_truth, recall_at_20, write_predictions


def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_interaction_matrix(data):
    rows = np.asarray(data.positive_users, dtype=np.int64)
    cols = np.asarray(data.positive_items, dtype=np.int64)
    mat = sp.coo_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                        shape=(data.num_users, data.num_items)).tocsr()
    mat.sum_duplicates()
    mat.data[:] = 1.0
    return mat


def ii_neighbors(mat, k: int):
    """Top-k co-occurrence cosine neighbors from FIT ONLY; padded weights=0."""
    n_items = mat.shape[1]
    ids = np.zeros((n_items, k), dtype=np.int64)
    weights = np.zeros((n_items, k), dtype=np.float32)
    deg = np.maximum(np.asarray(mat.getnnz(axis=0), dtype=np.float32), 1)
    co = (mat.T @ mat).tocsr()
    inv = np.sqrt(deg)
    for i in range(n_items):
        start, end = co.indptr[i], co.indptr[i+1]
        nbrs = co.indices[start:end]
        vals = co.data[start:end]
        if not len(nbrs):
            continue
        scores = vals / (inv[i] * inv[nbrs])
        keep = nbrs != i
        nbrs, scores = nbrs[keep], scores[keep]
        if not len(nbrs):
            continue
        take = min(k, len(nbrs))
        if take < len(nbrs):
            pos = np.argpartition(scores, -take)[-take:]
        else:
            pos = np.arange(len(nbrs))
        pos = pos[np.argsort(-scores[pos], kind='stable')]
        ids[i, :take] = nbrs[pos]
        values = scores[pos].astype(np.float32)
        weights[i, :take] = values / max(values.sum(), 1e-9)
    return ids, weights


class UltraStyle(nn.Module):
    def __init__(self, n_users, n_items, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.normal_(self.user_emb.weight, std=.05)
        nn.init.normal_(self.item_emb.weight, std=.05)

    def vectors(self, mat=None):
        return self.user_emb.weight, self.item_emb.weight


class SimpleXStyle(nn.Module):
    def __init__(self, n_users, n_items, dim, gamma):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        self.gamma = gamma
        nn.init.normal_(self.user_emb.weight, std=.05)
        nn.init.normal_(self.item_emb.weight, std=.05)

    def vectors(self, mat):
        # CSR FIT-history aggregation only; do not include held-out labels.
        history = torch.sparse_csr_tensor(
            torch.from_numpy(mat.indptr.astype(np.int64)).to(self.item_emb.weight.device),
            torch.from_numpy(mat.indices.astype(np.int64)).to(self.item_emb.weight.device),
            torch.from_numpy(mat.data.astype(np.float32)).to(self.item_emb.weight.device),
            mat.shape, device=self.item_emb.weight.device)
        lengths = torch.tensor(np.maximum(mat.getnnz(axis=1), 1),
                               dtype=torch.float32, device=self.item_emb.weight.device)
        agg = torch.sparse.mm(history, self.item_emb.weight) / lengths[:, None]
        return (F.normalize(self.user_emb.weight + self.gamma * agg, dim=-1),
                F.normalize(self.item_emb.weight, dim=-1))


def history_sum_for_batch(item_emb, flat_hist, rowptr, users):
    """Differentiable ragged history sum, for selected users only."""
    starts = rowptr[users]
    lengths = rowptr[users + 1] - starts
    offsets = torch.cat([torch.zeros((1,), device=users.device, dtype=torch.long),
                         lengths.cumsum(0)[:-1]])
    owner = torch.repeat_interleave(torch.arange(len(users), device=users.device), lengths)
    flat_positions = torch.repeat_interleave(starts, lengths) + (
        torch.arange(owner.numel(), device=users.device) - offsets[owner]
    )
    bag_idx = flat_hist[flat_positions]
    sums = F.embedding_bag(bag_idx, item_emb.weight, offsets,
                           mode='sum', include_last_offset=False)
    return sums, lengths


def sample_batch(users_all, items_all, n_items, observed_codes, batch, n_negs):
    dev = users_all.device
    sampled = torch.randint(len(users_all), (batch,), device=dev)
    users = users_all[sampled]
    positives = items_all[sampled]
    negatives = torch.randint(n_items, (batch, n_negs), device=dev)
    for _ in range(15):
        ids = users[:, None] * n_items + negatives
        pos = torch.searchsorted(observed_codes, ids.reshape(-1)).clamp_max(len(observed_codes)-1)
        mask = (observed_codes[pos].reshape_as(ids) == ids)
        if not mask.any().item():
            break
        negatives[mask] = torch.randint(n_items, (int(mask.sum().item()),), device=dev)
    else:
        raise RuntimeError('Unable to sample unobserved negatives')
    return users, positives, negatives


def evaluate(model, data, mat, truth, device, batch_size=128):
    model.eval()
    predictions = {}
    with torch.no_grad():
        user_vectors, item_vectors = model.vectors(mat)
        item_trans = item_vectors.T.contiguous()
        for start in range(0, len(data.target_users), batch_size):
            batch_users = data.target_users[start:start+batch_size]
            uidx = [data.user_index[u] for u in batch_users]
            scores = user_vectors[torch.tensor(uidx, device=device)] @ item_trans
            for r, ui in enumerate(uidx):
                seen = data.seen_items.get(ui, set())
                if seen:
                    scores[r, torch.tensor(list(seen), dtype=torch.long,device=device)] = -float('inf')
            top = scores.topk(20, dim=1, sorted=True).indices.cpu().tolist()
            for user, idxs in zip(batch_users, top):
                predictions[user] = [data.item_ids[i] for i in idxs]
        score = recall_at_20(predictions, truth) if truth else None
    return predictions, score


def run(a):
    assert a.epochs > 0 and a.eval_every > 0 and a.batch_size > 0 and a.steps_per_epoch > 0
    assert a.negatives > 0 and a.lr > 0 and a.reg >= 0
    device = torch.device(a.device)
    seed_all(a.seed)
    data = load_data(a.fit, a.users)
    truth = load_truth(a.truth, data.target_users) if a.truth else None
    mat = get_interaction_matrix(data)
    if a.model == 'ultragcn':
        model = UltraStyle(data.num_users, data.num_items, a.dim).to(device)
        user_degree = torch.tensor(np.maximum(mat.getnnz(axis=1), 1), dtype=torch.float32, device=device)
        item_degree = torch.tensor(np.maximum(mat.getnnz(axis=0), 1), dtype=torch.float32, device=device)
        if a.ii_weight > 0:
            ii_id, ii_wt = ii_neighbors(mat, a.ii_topk)
            ii_id = torch.from_numpy(ii_id).to(device)
            ii_wt = torch.from_numpy(ii_wt).to(device)
        else:
            ii_id, ii_wt = None, None
    elif a.model == 'simplex':
        model = SimpleXStyle(data.num_users, data.num_items, a.dim, a.gamma).to(device)
        rowptr = torch.tensor(mat.indptr.astype(np.int64), device=device)
        flat_history = torch.tensor(mat.indices.astype(np.int64), device=device)
    else:
        raise ValueError('Unexpected model')
    optim = torch.optim.Adam(model.parameters(), lr=a.lr)
    u_edges = torch.tensor(data.positive_users, dtype=torch.long, device=device)
    i_edges = torch.tensor(data.positive_items, dtype=torch.long, device=device)
    observed = (u_edges * data.num_items + i_edges).sort().values.unique(sorted=True)

    best_epoch = None
    best_score = -1.0
    best_state = None
    stale = 0
    last_loss = None
    for epoch in range(1, a.epochs+1):
        begin = time.time()
        model.train()
        loss_acc = 0.0
        for _ in range(a.steps_per_epoch):
            u, i, j = sample_batch(u_edges, i_edges, data.num_items, observed,
                                   a.batch_size, a.negatives)
            optim.zero_grad(set_to_none=True)
            if a.model == 'ultragcn':
                uv = model.user_emb(u)
                pv = model.item_emb(i)
                nv = model.item_emb(j)
                pos = (uv * pv).sum(-1)
                neg = (uv[:,None,:] * nv).sum(-1)
                # Degree-aware confidence for user-item constraint.
                confidence = (user_degree[u] / item_degree[i]).sqrt().clamp(0.25, 4.0)
                bce_pos = (1.0 + a.beta * confidence) * F.softplus(-pos)
                bce_neg = F.softplus(neg).mean(-1) * a.neg_weight
                loss = (bce_pos + bce_neg).mean()
                if ii_id is not None:
                    nbr_emb = model.item_emb(ii_id[i])
                    nbr_s = (uv[:,None,:] * nbr_emb).sum(-1)
                    item_loss = (F.softplus(-nbr_s) * ii_wt[i]).sum(-1).mean()
                    loss = loss + a.ii_weight * item_loss
                reg = .5*(uv.square().sum(-1).mean()+pv.square().sum(-1).mean()+nv.square().sum(-1).mean())
            else:
                # Leave the current target item OUT of the user's training history.
                history_sum, lengths = history_sum_for_batch(model.item_emb, flat_history, rowptr, u)
                agg_loo = (history_sum - model.item_emb(i)) / (lengths-1).clamp_min(1)[:,None]
                user_r = F.normalize(model.user_emb(u) + model.gamma * agg_loo, dim=-1)
                p = F.normalize(model.item_emb(i), dim=-1)
                n = F.normalize(model.item_emb(j), dim=-1)
                pos_s = (user_r * p).sum(-1)
                neg_s = (user_r[:,None,:] * n).sum(-1)
                loss = ((1.0 - pos_s) + a.neg_weight * F.relu(neg_s-a.margin).square().mean(-1)).mean()
                reg = .5*(model.user_emb(u).square().sum(-1).mean()+model.item_emb(i).square().sum(-1).mean()+model.item_emb(j).square().sum(-1).mean())
            loss = loss + a.reg * reg
            if not torch.isfinite(loss).item():
                raise RuntimeError('Nonfinite loss')
            loss.backward()
            optim.step()
            loss_acc += loss.item()
        last_loss = loss_acc / a.steps_per_epoch
        if (epoch % a.eval_every == 0 or epoch == a.epochs or epoch == 1) and truth:
            preds, result = evaluate(model, data, mat, truth, device, a.eval_batch)
            best = result['recall_at_20'] > best_score
            if best:
                best_score = result['recall_at_20']
                best_epoch = epoch
                best_state = copy.deepcopy({k:v.detach().cpu() for k,v in model.state_dict().items()})
                stale = 0
            else:
                stale += 1
            print(json.dumps({'epoch':epoch,'loss':last_loss,'recall_at_20':result['recall_at_20'],
                              'hits':result['hits'],'best':best,'seconds':round(time.time()-begin,2)}),flush=True)
            if a.patience>0 and stale>=a.patience:
                break
        elif epoch % max(a.eval_every, 10) == 0 or epoch == a.epochs:
            print(json.dumps({'epoch':epoch,'loss':last_loss,'seconds':round(time.time()-begin,2)}),flush=True)

    if best_state is not None:
        model.load_state_dict(best_state)
    predictions, result = evaluate(model, data, mat, truth, device, a.eval_batch)
    write_predictions(a.out, data.target_users, predictions)
    output = dict(model=a.model, out=a.out, fit=a.fit, best_epoch=best_epoch,
                  trained_epochs=epoch, epochs_requested=a.epochs, last_loss=last_loss,
                  dim=a.dim, lr=a.lr, negatives=a.negatives, reg=a.reg,
                  steps_per_epoch=a.steps_per_epoch, batch_size=a.batch_size,
                  ii_weight=a.ii_weight, beta=a.beta, gamma=a.gamma,
                  margin=a.margin, neg_weight=a.neg_weight, seed=a.seed)
    if result:
        output.update(result)
    if a.metrics:
        Path(a.metrics).write_text(json.dumps(output,indent=2),encoding='utf-8')
    print(json.dumps({'final':output}), flush=True)
    return output


def main():
    p=argparse.ArgumentParser(description='UltraGCN-style and SimpleX-style collaborative filtering')
    p.add_argument('--model', choices=['ultragcn','simplex'], required=True)
    p.add_argument('--fit', required=True)
    p.add_argument('--users', required=True)
    p.add_argument('--truth', default=None)
    p.add_argument('--out', required=True)
    p.add_argument('--metrics', default=None)
    p.add_argument('--epochs',type=int,default=70)
    p.add_argument('--eval-every',type=int,default=10)
    p.add_argument('--patience',type=int,default=5)
    p.add_argument('--device',default='cuda:0')
    p.add_argument('--dim',type=int,default=128)
    p.add_argument('--batch-size',type=int,default=2048)
    p.add_argument('--steps-per-epoch',type=int,default=40)
    p.add_argument('--negatives',type=int,default=10)
    p.add_argument('--lr',type=float,default=.002)
    p.add_argument('--reg',type=float,default=1e-5)
    p.add_argument('--ii-weight',type=float,default=.1)
    p.add_argument('--ii-topk',type=int,default=10)
    p.add_argument('--beta',type=float,default=.1)
    p.add_argument('--neg-weight',type=float,default=4.)
    p.add_argument('--gamma',type=float,default=1.)
    p.add_argument('--margin',type=float,default=.2)
    p.add_argument('--seed',type=int,default=42)
    p.add_argument('--eval-batch',type=int,default=128)
    a=p.parse_args()
    run(a)


if __name__=='__main__':
    main()


## 3. 동일 팀 Validation ZIP 연결 및 데이터 해시 검사

In [ ]:
sys.path.insert(0, str(WORK))
from teamkit import verify_split, read_truth

required = ('val_train.csv','val_truth.csv','val_users.csv','manifest.json')
source_folder = None
source_zip = None
for base in (INPUT, WORK):
    for mf in base.rglob('manifest.json'):
        folder = mf.parent
        if folder == VALID or not all((folder/n).is_file() for n in required):
            continue
        try:
            obj = json.loads(mf.read_text(encoding='utf-8'))
        except (OSError,ValueError):
            continue
        if obj.get('split_id') == SPLIT_ID:
            source_folder = folder
            break
    if source_folder:
        break
if not source_folder:
    for base in (INPUT, WORK):
        hits = list(base.rglob('amazon_topk_teamkit_20261110.zip'))
        if hits:
            source_zip = hits[0]
            break
if source_folder is None and source_zip is None:
    raise FileNotFoundError('기존 팀 공통 ZIP을 오른쪽 Input에서 Add Input으로 연결해 주세요.')
VALID.mkdir(parents=True, exist_ok=True)
if source_folder is not None:
    for n in required:
        shutil.copyfile(source_folder/n, VALID/n)
    print('Team ZIP read from:', source_folder)
else:
    with zipfile.ZipFile(source_zip) as z:
        for n in required:
            paths = [x for x in z.namelist() if x.endswith('/validation/'+n)]
            if len(paths) != 1:
                raise ValueError(f'공통 ZIP 파일 {n}을 정확히 한 개 찾지 못했습니다: {paths}')
            with z.open(paths[0]) as src, (VALID/n).open('wb') as dst:
                shutil.copyfileobj(src, dst)
    print('Team ZIP extracted from:', source_zip)
manifest = verify_split(VALID)
assert manifest['split_id'] == SPLIT_ID, '팀 공통 split ID가 다릅니다.'
h = hashlib.sha256()
with TRAIN.open('rb') as f:
    for chunk in iter(lambda:f.read(1024*1024), b''):
        h.update(chunk)
assert h.hexdigest() == manifest['source_train_sha256'], '원본 train.csv가 팀 공통 분할과 일치하지 않습니다.'
print('Validation matched:', manifest['validation_users'],'users,',manifest['fit_rows'],'train edges,',manifest['truth_rows'],'truth edges')
print('Split ID:',manifest['split_id'])

## 4. Ablation 설정

기본 `BALANCED`는 10개의 새 학습 실험입니다. 기존 4개 알고리즘은 동일 예측을 확보하기 위해 기본적으로 재학습합니다. 다른 노트북에서 얻은 `*_val.csv`가 있더라도 **새 노트북 실행환경에 자동으로 전달되지 않습니다**. 이전 실험 노트북의 `Output`을 별도 Dataset으로 연결하는 선택지는 아래 코드 주석에서 확인하세요.

**Epoch 설정은 `최대값`입니다.** 개선이 장시간 없으면 Early Stopping으로 먼저 끝납니다. SimpleX 평가는 25 Epoch 간격, UltraGCN은 10 Epoch 간격입니다. UltraGCN의 초기 최고 시점을 놓치지 않기 위한 설정입니다.


In [ ]:
RUN_PROFILE = 'BALANCED'       # 'QUICK' (6) | 'BALANCED' (10, 권장) | 'EXTENDED' (16)
RETRAIN_EXISTING = True        # LightGCN, NGCF, SGL, NCL을 동일 split으로 다시 예측
SKIP_COMPLETED = True         # 동일 설정/코드의 완료된 결과만 재사용
MAKE_SUBMISSION = False       # 이 노트북은 Validation 전용
ENSEMBLE_TOP_SAVE = 5         # 상위 앙상블의 예측만 CSV 저장(파일 용량 절약)

BASE_SPECS = {
  'lightgcn': dict(name='lightgcn', model='lightgcn', runner='graph', epochs=575,
                   dim=128, layers=3, lr=.003, reg=.0001),
  'ngcf': dict(name='ngcf', model='ngcf', runner='graph', epochs=800,
               dim=64, layers=2, lr=.001, reg=.0001, message_dropout=.1),
  'sgl': dict(name='sgl', model='sgl', runner='ssl', epochs=525,
              dim=128, layers=3, lr=.003, reg=.0001,
              ssl_weight=.0005, drop=.1, tau=.2),
  'ncl': dict(name='ncl', model='ncl', runner='ssl', epochs=425,
              dim=128, layers=3, lr=.003, reg=.0001,
              ssl_weight=.001, proto_weight=0., tau=.2),
}

# 초반 실험의 최고 설정 Ultra U2: ii_weight=.30, negatives=10 (best_epoch=40)
# SimpleX X1: gamma=1.0, margin=.20, negatives=20 (best_epoch=70, 계속 개선 중)
# 단일요인 변화 중심. 동일 모델의 steps_per_epoch=35 고정해 비교 가능하도록 설계
ULTRA_SPECS = [
  dict(name='U00_ref_long',      model='ultragcn',runner='new',epochs=350, dim=128,lr=.002,reg=.00001, negatives=10,ii_weight=.30,beta=.1,neg_weight=4.,steps_per_epoch=35,eval_every=10,patience=15),
  dict(name='U01_ii015',         model='ultragcn',runner='new',epochs=350, dim=128,lr=.002,reg=.00001, negatives=10,ii_weight=.15,beta=.1,neg_weight=4.,steps_per_epoch=35,eval_every=10,patience=15),
  dict(name='U02_ii050',         model='ultragcn',runner='new',epochs=350, dim=128,lr=.002,reg=.00001, negatives=10,ii_weight=.50,beta=.1,neg_weight=4.,steps_per_epoch=35,eval_every=10,patience=15),
  dict(name='U03_neg20',         model='ultragcn',runner='new',epochs=350, dim=128,lr=.002,reg=.00001, negatives=20,ii_weight=.30,beta=.1,neg_weight=4.,steps_per_epoch=35,eval_every=10,patience=15),
  dict(name='U04_lr001',         model='ultragcn',runner='new',epochs=450, dim=128,lr=.001,reg=.00001, negatives=10,ii_weight=.30,beta=.1,neg_weight=4.,steps_per_epoch=35,eval_every=10,patience=18),
  dict(name='U05_beta050',       model='ultragcn',runner='new',epochs=350, dim=128,lr=.002,reg=.00001, negatives=10,ii_weight=.30,beta=.5,neg_weight=4.,steps_per_epoch=35,eval_every=10,patience=15),
  dict(name='U06_dim64',         model='ultragcn',runner='new',epochs=450, dim=64, lr=.002,reg=.00001, negatives=10,ii_weight=.30,beta=.1,neg_weight=4.,steps_per_epoch=35,eval_every=10,patience=18),
  dict(name='U07_negweight2',    model='ultragcn',runner='new',epochs=350, dim=128,lr=.002,reg=.00001, negatives=10,ii_weight=.30,beta=.1,neg_weight=2.,steps_per_epoch=35,eval_every=10,patience=15),
]
SIMPLEX_SPECS = [
  dict(name='X00_ref_700',       model='simplex',runner='new',epochs=700,dim=128,lr=.002,reg=.00001,negatives=20,gamma=1., margin=.20,neg_weight=4.,steps_per_epoch=35,eval_every=25,patience=12),
  dict(name='X01_gamma05_700',   model='simplex',runner='new',epochs=700,dim=128,lr=.002,reg=.00001,negatives=20,gamma=.5, margin=.20,neg_weight=4.,steps_per_epoch=35,eval_every=25,patience=12),
  dict(name='X02_gamma15_700',   model='simplex',runner='new',epochs=700,dim=128,lr=.002,reg=.00001,negatives=20,gamma=1.5,margin=.20,neg_weight=4.,steps_per_epoch=35,eval_every=25,patience=12),
  dict(name='X03_neg40_700',     model='simplex',runner='new',epochs=700,dim=128,lr=.002,reg=.00001,negatives=40,gamma=1., margin=.20,neg_weight=4.,steps_per_epoch=35,eval_every=25,patience=12),
  dict(name='X04_lr001_900',     model='simplex',runner='new',epochs=900,dim=128,lr=.001,reg=.00001,negatives=20,gamma=1., margin=.20,neg_weight=4.,steps_per_epoch=35,eval_every=25,patience=12),
  dict(name='X05_margin010_700', model='simplex',runner='new',epochs=700,dim=128,lr=.002,reg=.00001,negatives=20,gamma=1., margin=.10,neg_weight=4.,steps_per_epoch=35,eval_every=25,patience=12),
  dict(name='X06_dim256_800',    model='simplex',runner='new',epochs=800,dim=256,lr=.0015,reg=.00001,negatives=20,gamma=1.,margin=.20,neg_weight=4.,steps_per_epoch=35,eval_every=25,patience=12),
  dict(name='X07_lr003_700',     model='simplex',runner='new',epochs=700,dim=128,lr=.003,reg=.00001,negatives=20,gamma=1., margin=.20,neg_weight=4.,steps_per_epoch=35,eval_every=25,patience=12),
]

if RUN_PROFILE == 'QUICK':
    ULTRA_SPECS = ULTRA_SPECS[:3]
    SIMPLEX_SPECS = SIMPLEX_SPECS[:3]
elif RUN_PROFILE == 'BALANCED':
    ULTRA_SPECS = ULTRA_SPECS[:5]
    SIMPLEX_SPECS = SIMPLEX_SPECS[:5]
elif RUN_PROFILE != 'EXTENDED':
    raise ValueError('RUN_PROFILE은 QUICK / BALANCED / EXTENDED 중 선택')
NEW_SPECS = ULTRA_SPECS + SIMPLEX_SPECS
assert not MAKE_SUBMISSION
assert len({s['name'] for s in [*BASE_SPECS.values(), *NEW_SPECS]}) == len(BASE_SPECS)+len(NEW_SPECS)
print('신규 실험:',len(NEW_SPECS),'= UltraGCN',len(ULTRA_SPECS),'+ SimpleX',len(SIMPLEX_SPECS))
print('SIMPLEX 최대 Epoch:',[s['epochs'] for s in SIMPLEX_SPECS])
print('Ultra 최대 Epoch:',[s['epochs'] for s in ULTRA_SPECS])
print('기존 4개 모델 재생성:',RETRAIN_EXISTING,'| 제출:',MAKE_SUBMISSION)

## 5. 공통 학습·검증·캐시 코드

`RUN_PROFILE`을 EXTENDED로 바꿔서 다시 실행할 때는 **기존 실행된 결과 파일이 남아있는 동일 Kaggle 실행/저장 버전**이라면 설정 및 소스 해시가 일치하는 결과만 재사용합니다. 새로운 노트북 세션에서는 파일이 보존되지 않을 수 있으니 `Save Version` 또는 출력 파일 저장을 확인하세요.


In [ ]:
from teamkit import read_interactions, read_truth, read_predictions, validate_predictions, recall_at_20
from graph_models import write_predictions
from collections import defaultdict

FIT = VALID/'val_train.csv'
VAL_USERS = VALID/'val_users.csv'
TRUTH = VALID/'val_truth.csv'
seen, popularity = read_interactions(FIT)
truth = read_truth(TRUTH)
all_catalog = set(popularity) | {item for items in truth.values() for item in items}
RESULTS = {}
FAILURES = {}

SOURCES = ['teamkit.py','graph_models.py','sgl_ncl_runner.py','ultra_simplex_runner.py']
SOURCE_SHA = {name:hashlib.sha256((WORK/name).read_bytes()).hexdigest() for name in SOURCES}


def build_cmd(spec, dest, metric, fit=FIT, users=VAL_USERS, truth_file=TRUTH):
    common = ['--fit',str(fit),'--users',str(users),'--out',str(dest),
              '--epochs',str(spec['epochs']),'--lr',str(spec['lr']), '--reg',str(spec['reg']),
              '--seed',str(SEED),'--device',DEVICE]
    if spec['runner'] == 'graph':
        cmd = [sys.executable,str(WORK/'graph_models.py'), '--model',spec['model'],
               *common, '--metrics-out',str(metric),
               '--embedding-dim',str(spec['dim']), '--layers',str(spec['layers']),
               '--samples-per-epoch','100000', '--eval-every','25', '--patience','0']
        if spec['model']=='ngcf':cmd += ['--message-dropout',str(spec.get('message_dropout',.1))]
    elif spec['runner'] == 'ssl':
        cmd = [sys.executable,str(WORK/'sgl_ncl_runner.py'), '--model',spec['model'],
               *common, '--metrics',str(metric),
               '--dim',str(spec['dim']), '--layers',str(spec['layers']),
               '--samples-per-epoch','100000','--eval-every','25','--patience','0',
               '--ssl-weight',str(spec.get('ssl_weight',.005)),
               '--proto-weight',str(spec.get('proto_weight',0.)),
               '--drop',str(spec.get('drop',.1)),'--tau',str(spec.get('tau',.2)),
               '--ssl-nodes','192','--warmup','100']
    else:
        cmd = [sys.executable,str(WORK/'ultra_simplex_runner.py'), '--model',spec['model'],
               *common,'--metrics',str(metric),'--dim',str(spec['dim']),
               '--batch-size','2048','--steps-per-epoch',str(spec['steps_per_epoch']),
               '--negatives',str(spec['negatives']),
               '--ii-weight',str(spec.get('ii_weight',0.)),
               '--beta',str(spec.get('beta',.1)),'--neg-weight',str(spec.get('neg_weight',4.)),
               '--gamma',str(spec.get('gamma',1.)), '--margin',str(spec.get('margin',.2)),
               '--eval-every',str(spec.get('eval_every',25)),
               '--patience',str(spec.get('patience',12))]
    if truth_file is not None:cmd += ['--truth',str(truth_file)]
    return cmd


def train_and_validate(spec):
    name = spec['name']
    dest = OUT/f'{name}_val.csv'
    metric = OUT/f'{name}.metrics.json'
    stamp = OUT/f'{name}.config.json'
    cfg = {'spec':spec, 'split_id':SPLIT_ID,
           'source_train_sha256':manifest['source_train_sha256'],
           'code_sha256':SOURCE_SHA, 'fit_file':str(FIT)}
    if SKIP_COMPLETED and dest.is_file() and metric.is_file() and stamp.is_file():
        try:
            if json.loads(stamp.read_text(encoding='utf-8')) == cfg:
                pred=read_predictions(dest)
                validate_predictions(pred, truth, seen, all_catalog)
                score=recall_at_20(pred, truth)
                info=json.loads(metric.read_text(encoding='utf-8'))
                if abs(float(info['recall_at_20'])-score['recall_at_20']) < 1e-12:
                    print('CACHE HIT:',name,'Recall@20=',round(score['recall_at_20'],7),flush=True)
                    return info
        except Exception as exc:
            print('Cache invalid:',name,repr(exc),flush=True)
    dest.unlink(missing_ok=True)
    metric.unlink(missing_ok=True)
    cmd=build_cmd(spec,dest,metric)
    log=OUT/f'{name}.log'
    print('START',name,'max_epoch=',spec['epochs'],flush=True)
    with log.open('w',encoding='utf-8') as f:
        proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1)
        for line in proc.stdout:
            f.write(line)
            if ('"recall_at_20"' in line or 'Traceback' in line or 'Error' in line or 'OOM' in line):
                print(line.rstrip()[:550],flush=True)
        rc=proc.wait()
    if rc != 0:
        raise RuntimeError(f'{name} exit={rc} / 상세 로그: {log}')
    if not dest.is_file() or not metric.is_file():
        raise RuntimeError(f'{name}: 학습 종료 후 CSV 또는 메트릭이 생성되지 않았습니다.')
    pred=read_predictions(dest)
    validate_predictions(pred,truth,seen,all_catalog)
    score=recall_at_20(pred,truth)
    info=json.loads(metric.read_text(encoding='utf-8'))
    if abs(float(info['recall_at_20']) - score['recall_at_20']) > 1e-12:
        raise RuntimeError(f'{name} 메트릭과 CSV 평가 결과 불일치')
    stamp.write_text(json.dumps(cfg,indent=2,ensure_ascii=False),encoding='utf-8')
    print('DONE',name,'Recall@20=',round(score['recall_at_20'],7),
          '| best=',info['best_epoch'],'| trained=',info['trained_epochs'],flush=True)
    return info


def record(spec,info):
    RESULTS[spec['name']] = dict(name=spec['name'], model=spec['model'], runner=spec['runner'],
                                  score=float(info['recall_at_20']), hits=int(info['hits']),
                                  best_epoch=int(info['best_epoch']),
                                  trained_epochs=int(info['trained_epochs']), spec=spec)
    (OUT/'single_results.json').write_text(
        json.dumps(list(RESULTS.values()),indent=2,ensure_ascii=False),encoding='utf-8')


## 6. UltraGCN: 3~8가지 Ablation

**기존 U2 최고**: Recall@20 ≈ 0.027211 / best_epoch=40. 이번에는 item-item 가중치, negative 수, 학습률, beta 등을 비교하며 길게 학습해도 성능이 올라가는지 검사합니다. 개선이 멈추면 조기 종료합니다.


In [ ]:
for j,s in enumerate(ULTRA_SPECS,1):
    print(f'[{j}/{len(ULTRA_SPECS)}] UltraGCN: {s["name"]}',flush=True)
    try:
        record(s,train_and_validate(s))
    except Exception as exc:
        FAILURES[s['name']] = repr(exc)
        print('FAIL:',s['name'],repr(exc),flush=True)
(OUT/'failures.json').write_text(json.dumps(FAILURES,ensure_ascii=False,indent=2))
urows=[{k:v for k,v in r.items() if k!='spec'} for r in RESULTS.values() if r['model']=='ultragcn']
if urows:display(pd.DataFrame(urows).sort_values('score',ascending=False))
else:print('주의: UltraGCN 성공 없음. failures.json 확인')

## 7. SimpleX: 3~8가지 Long-Horizon Ablation

**기존 X1 최고**: Recall@20 ≈ 0.030465 / best_epoch=70 (실험 종료 시점). 이제 최대 700~900 Epoch까지 학습하고, 이력 가중치 gamma, negative 수, LR, margin을 비교합니다. 조기 종료 조건은 Validation 25 Epoch 간격에서 12번 연속 무개선입니다.


In [ ]:
for j,s in enumerate(SIMPLEX_SPECS,1):
    print(f'[{j}/{len(SIMPLEX_SPECS)}] SimpleX: {s["name"]}',flush=True)
    try:
        record(s,train_and_validate(s))
    except Exception as exc:
        FAILURES[s['name']] = repr(exc)
        print('FAIL:',s['name'],repr(exc),flush=True)
(OUT/'failures.json').write_text(json.dumps(FAILURES,ensure_ascii=False,indent=2))
xrows=[{k:v for k,v in r.items() if k!='spec'} for r in RESULTS.values() if r['model']=='simplex']
if xrows:display(pd.DataFrame(xrows).sort_values('score',ascending=False))
else:print('주의: SimpleX 성공 없음. failures.json 확인')

## 8. 기존 4개 모델 확보 (LightGCN·NGCF·SGL·NCL)

가장 정확한 비교는 **동일한 팀 Validation split**에서 기존 모델을 재학습하는 것입니다. 새 Kaggle 노트북에서 이전 노트북의 메모리나 파일은 자동으로 공유되지 않기 때문에 기본은 재학습입니다. 완료된 설정은 동일 세션에서 재실행 시 캐시로 건너뜁니다.


In [ ]:
if not RETRAIN_EXISTING:
    raise ValueError('RETRAIN_EXISTING=False는 예측 파일을 직접 읽는 사용자 확장 모드입니다. 기본 True로 실행해 주세요.')
for idx,(model,spec) in enumerate(BASE_SPECS.items(),1):
    print(f'[{idx}/4] 기존 모델 {model}:',flush=True)
    try:
        record(spec,train_and_validate(spec))
    except Exception as exc:
        FAILURES[spec['name']]=repr(exc)
        print('FAIL:',spec['name'],repr(exc),flush=True)
(OUT/'failures.json').write_text(json.dumps(FAILURES,ensure_ascii=False,indent=2))
print('기존 모델 완료. 실패 항목:',FAILURES)

## 9. 단일 모델 선택 및 Top-20 예측 로드

6개 알고리즘의 각 1위 설정을 선택합니다. **SimpleX 상위 2설정**은 후속 LightGCN 앙상블에 각각 사용하므로, 단일 성능 2위 SimpleX가 앙상블 1위가 될 가능성도 검사합니다.


In [ ]:
def ranked_model(model):
    return sorted((r for r in RESULTS.values() if r['model']==model),
                  key=lambda r:(-r['score'],r['name']))

required_models = ['lightgcn','ngcf','sgl','ncl','ultragcn','simplex']
missing=[m for m in required_models if not ranked_model(m)]
if missing:
    raise RuntimeError('6개 모델을 모두 비교하려면 실패한 모델을 해결해 주세요: '+str(missing))
SIX={m:ranked_model(m)[0] for m in required_models}
print('=== 6개 모델 최고 설정 ===')
for m,r in SIX.items():
    print(f'{m:9s} {r["name"]:24s} score={r["score"]:.6f} best_epoch={r["best_epoch"]}')

PREDICTIONS={m:read_predictions(OUT/f'{r["name"]}_val.csv') for m,r in SIX.items()}
# SimpleX 상위 2개 각각 사용: 새 설정들의 상보성이 중요
SX2=ranked_model('simplex')[:2]
for j,r in enumerate(SX2):
    key=f'simplex_top{j+1}'
    PREDICTIONS[key]=read_predictions(OUT/f'{r["name"]}_val.csv')
for m,pred in PREDICTIONS.items():
    validate_predictions(pred,truth,seen,all_catalog)
print('검증된 예측 목록:',list(PREDICTIONS))

## 10. 앙상블 Ablation — RRF k, 가중치, 구성모델 비교

6개 모델 모두를 **다시 학습하지 않고 이미 만든 추천 리스트만** 결합합니다. 이전 최고 기록 `LightGCN + SimpleX RRF = 0.037879`를 넘는지 비교합니다.

- LightGCN+SimpleX **상위 2개 각각**: RRF k=20/60 × 가중치 L:X=25:75/50:50/75:25 → 12개
- LightGCN+UltraGCN: RRF k=20/60 × 2개 가중치 → 4개
- SimpleX+UltraGCN, LightGCN+SimpleX+UltraGCN, SGL/NCL 포함, 6개 전체, Rank(Borda) 조합.

Top-20 리스트만 활용하므로 *추천에 등장하지 않은 아이템은 앙상블에 추가되지 않습니다*. 원시 점수 규모가 다른 모델을 직접 합치지 않습니다.


In [ ]:
from collections import defaultdict

ENSEMBLE_CONFIGS=[]
def add_ens(name,members,weights,method='rrf',k=60):
    assert len(members)==len(weights) and len(members)>=2
    assert all(m in PREDICTIONS for m in members)
    ENSEMBLE_CONFIGS.append(dict(name=name,members=members,weights=weights,method=method,k=k))

# L + X top1/top2, 3 weights x 2 k = 12
for sx in ['simplex_top1','simplex_top2'][:len(SX2)]:
    for k in (20,60):
        for lweight in (.25,.50,.75):
            add_ens(f'E_L_{sx}_k{k}_L{int(lweight*100)}',
                    ['lightgcn',sx],[lweight,1-lweight],k=k)
# L + U (4)
for k in (20,60):
    for lw in (.5,.7):
        add_ens(f'E_L_U_k{k}_L{int(lw*100)}',['lightgcn','ultragcn'],[lw,1-lw],k=k)
# X + U (2)
for k in (20,60):
    add_ens(f'E_X_U_k{k}', ['simplex','ultragcn'],[.5,.5],k=k)
# L + X + U (4)
for k in (20,60):
    add_ens(f'E_L_X_U_k{k}_a',['lightgcn','simplex','ultragcn'],[.5,.35,.15],k=k)
    add_ens(f'E_L_X_U_k{k}_b',['lightgcn','simplex','ultragcn'],[.35,.5,.15],k=k)
# L + X + structural auxiliaries (2)
add_ens('E_L_X_SGL',['lightgcn','simplex','sgl'],[.4,.4,.2])
add_ens('E_L_X_NCL',['lightgcn','simplex','ncl'],[.4,.4,.2])
# all 6, 2 weights
add_ens('E_ALL6_EQUAL',required_models,[1]*6)
add_ens('E_ALL6_FOCUS_LX',required_models,[3,1,1,1,1,3])
# Borda/Rank comparison (2)
add_ens('E_L_X_RANK_5050',['lightgcn','simplex'],[.5,.5],method='rank')
add_ens('E_L_X_RANK_7030',['lightgcn','simplex'],[.7,.3],method='rank')

assert len({c['name'] for c in ENSEMBLE_CONFIGS})==len(ENSEMBLE_CONFIGS)

TARGET_USERS=list(truth)
def _iid_key(item):
    return (0,int(item)) if item.isdecimal() else (1,item)

def fuse_user(user,cfg):
    scores=defaultdict(float)
    for name,w in zip(cfg['members'],cfg['weights']):
        for rank,item in enumerate(PREDICTIONS[name][user],1):
            if cfg['method']=='rrf':
                scores[item]+=float(w)/(int(cfg['k'])+rank)
            elif cfg['method']=='rank':
                scores[item]+=float(w)*(21-rank)
            else:raise ValueError(cfg['method'])
    return sorted(scores,key=lambda item:(-scores[item],_iid_key(item)))[:20]

def fuse_all(cfg):
    return {user:fuse_user(user,cfg) for user in TARGET_USERS}

ENSEMBLE_RESULTS={}
for i,cfg in enumerate(ENSEMBLE_CONFIGS,1):
    pp=fuse_all(cfg)
    validate_predictions(pp,truth,seen,all_catalog)
    metrics=recall_at_20(pp,truth)
    ENSEMBLE_RESULTS[cfg['name']]={**cfg,'score':metrics['recall_at_20'],'hits':int(metrics['hits'])}
    print(f'[{i:02}/{len(ENSEMBLE_CONFIGS)}] {cfg["name"]:29s} recall={metrics["recall_at_20"]:.6f}',flush=True)
    del pp
(OUT/'ensemble_results.json').write_text(json.dumps(list(ENSEMBLE_RESULTS.values()),indent=2,ensure_ascii=False))
print('앙상블 수:',len(ENSEMBLE_RESULTS))

## 11. 전체 모델+앙상블 순위 / Top 5 저장

지나치게 많은 CSV가 생기지 않게 **모든 Ablation의 점수표를 저장하되, 앙상블 예측 CSV는 상위 5개만** 보관합니다. `comparison.csv`가 핵심 파일입니다. 기존 최고값 0.037879보다 얼마나 좋아졌는지도 확인합니다.


In [ ]:
single_rows=[dict(name=r['name'],kind='single',models=r['model'],score=r['score'],
                  hits=r['hits'],best_epoch=r['best_epoch'],trained_epochs=r['trained_epochs'])
             for r in RESULTS.values()]
ensemble_rows=[dict(name=e['name'],kind='ensemble',models='+'.join(e['members']),
                    score=e['score'],hits=e['hits'],best_epoch=None,trained_epochs=None)
               for e in ENSEMBLE_RESULTS.values()]
comparison=pd.DataFrame(single_rows+ensemble_rows).sort_values(['score','name'],ascending=[False,True])
comparison=comparison.reset_index(drop=True)
comparison.to_csv(OUT/'comparison.csv',index=False)
print('=== 성능 TOP 15 ===')
display(comparison.head(15))
print('=== 모델별 최고 단일 점수 ===')
display(pd.DataFrame([{k:v for k,v in r.items() if k!='spec'} for r in SIX.values()]).sort_values('score',ascending=False))
print('이전 최고 앙상블: 0.037879 | 이번 최고:',round(float(comparison.iloc[0]['score']),6),
      '| 차이:',round(float(comparison.iloc[0]['score'])-.037879,7))

for cfg in sorted(ENSEMBLE_CONFIGS,key=lambda c:-ENSEMBLE_RESULTS[c['name']]['score'])[:ENSEMBLE_TOP_SAVE]:
    pred=fuse_all(cfg)
    out_path=OUT/f'{cfg["name"]}_val.csv'
    write_predictions(out_path,TARGET_USERS,pred)
    print('Saved ensemble validation predictions:',out_path.name)

(OUT/'recommended_top5.json').write_text(
    comparison.head(5).to_json(orient='records',force_ascii=False,indent=2),encoding='utf-8')

if FAILURES:
    print('실패한 개별 실험이 있습니다:', FAILURES)
print('제출 파일 생성 여부:',MAKE_SUBMISSION,'(False = 제출하지 않음)')

## 12. 결과 ZIP 생성

Kaggle Notebook Output에서 `ultra_simplex_medium_results.zip`을 다운로드한 다음, `comparison.csv` 및 `single_results.json`/`ensemble_results.json`을 보내주세요. 원본 train/test와 Validation truth는 ZIP에 포함하지 않습니다.

**이번 코드는 제출 CSV를 만들지 않습니다.** 최고 후보 2~3개를 비교한 뒤 전체 train으로 재학습해 제출할 모델을 선택하는 별도 단계를 진행하면 됩니다.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
ZIP=WORK/'ultra_simplex_medium_results.zip'
with ZipFile(ZIP,'w',ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file():
            z.write(path,arcname=path.name)
print('ZIP READY:',ZIP,'| MB:',round(ZIP.stat().st_size/1024/1024,2))
print('Scoreboard:',OUT/'comparison.csv')
print('Submission 생성 안 함!')